In [ ]:
import numpy as np
import torch
from transformers import AutoImageProcessor, AutoModel
from qdrant_client import QdrantClient, models
import pandas as pd
from PIL import Image
import os
import time

os.getpid()

/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5003

In [2]:
#поднимаем ВБД с маппингом в корень прокта
# в корне проекта выполнить 

#docker run -p 6333:6333 -p 6334:6334 -v "$(pwd)/qdrant_storage:/qdrant/storage" qdrant/qdrant

# инициализация клиента ВБД
client = QdrantClient("http://localhost:6333")

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
prefix_path = "/home/inna/Рабочий стол/SneakerSearch/data/"

lamoda_data = pd.read_csv(prefix_path+"lamoda_data.csv", sep=";")
lamoda_data

/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


,brand,model,category,color,description,lamoda_photo,title_photo,path_to_lamoda_photo
0,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
1,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
2,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
3,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
4,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
...,...,...,...,...,...,...,...,...
2515,Makfine,Makfine Кроссовки,Низкие кроссовки,71956,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1KSX2_33...,lamoda_photos/Makfine_Makfine_Кроссовки__71956...,//a.lmcdn.ru/product/M/P/MP002XW1KSX2_33126398...
2516,Makfine,Makfine Кроссовки,Низкие кроссовки,71956,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1KSX2_33...,lamoda_photos/Makfine_Makfine_Кроссовки__71956...,//a.lmcdn.ru/product/M/P/MP002XW1KSX2_33126398...
2517,Makfine,Makfine Кроссовки,Низкие кроссовки,71956,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1KSX2_33...,lamoda_photos/Makfine_Makfine_Кроссовки__71956...,//a.lmcdn.ru/product/M/P/MP002XW1KSX2_33126398...
2518,Makfine,Makfine Кроссовки,Низкие кроссовки,71956,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1KSX2_33...,lamoda_photos/Makfine_Makfine_Кроссовки__71956...,//a.lmcdn.ru/product/M/P/MP002XW1KSX2_33126398...


In [4]:
# удаляем дубликаты
lamoda_data = lamoda_data.drop_duplicates(subset="title_photo")
print(len(lamoda_data))

2408


In [5]:
lamoda_data.iloc[3]["path_to_lamoda_photo"] == lamoda_data.iloc[6]["path_to_lamoda_photo"]

False

In [6]:
lamoda_data = lamoda_data.drop_duplicates(subset="path_to_lamoda_photo")
print(len(lamoda_data))

395


In [7]:
def create_db(
        client: QdrantClient,
        collection_name: str, 
        emb_dim: int,
        data: pd.DataFrame,
        get_image_embedding: callable):
    
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                 "photos": models.VectorParams(size=emb_dim, distance=models.Distance.COSINE)
            },
        )
        unprocessable = 0

        for idx, row in data.iterrows():
                img_path = prefix_path+lamoda_data.loc[idx]["title_photo"]
                try:
                    image_emb = get_image_embedding(img_path)

                    point = models.PointStruct(
                        id=idx, 
                        vector={
                             "photos" : image_emb.tolist()
                            }, 
                        payload={
                            "brand": row["brand"],
                            "model": row["model"],
                            "color": row["color"],
                            "path_to_photo": os.path.join(prefix_path, row["title_photo"]) # cохраняем путь для отображения!
                        }
                    )

                    client.upsert(
                        collection_name=collection_name, 
                        points=[point]
                        )
                except:
                    unprocessable+=1 
                    continue
        print("unprocessable: ", unprocessable)
    else:
        print("Коллекция уже существует")

In [8]:
user_data = pd.read_csv(prefix_path+"user_data.csv", sep=";")
user_data = user_data.drop_duplicates()
user_data = user_data.drop_duplicates(subset="path_to_user_photo")
user_data = user_data.sample(frac=1, random_state=42).reset_index(drop=True) #shuffle data
print(len(user_data))

464


In [9]:
user_data

,brand,model,category,color,description,title_photo,path_to_title_photo,user_photo,path_to_user_photo
0,U.S. Polo Assn.,U.S. Polo Assn. Кеды Полнота F (6),Низкие кеды,82356,NaN,//a.lmcdn.ru/product/M/P/MP002XW1KGE5_32904702...,lamoda_photos/U.S._Polo_Assn._U.S._Polo_Assn._...,https://a.lmcdn.ru/photoreview/?key=d20b6206-2...,user_photos/U.S._Polo_Assn._U.S._Polo_Assn._Ке...
1,Baasploa,Baasploa Кроссовки,Низкие кроссовки,63085,,//a.lmcdn.ru/product/M/P/MP002XW0OZFJ_22116294...,lamoda_photos/Baasploa_Baasploa_Кроссовки__0.jpg,https://a.lmcdn.ru/photoreview/?key=80011863-a...,user_photos/Baasploa_Baasploa_Кроссовки__2.jpg
2,Covani,Covani Кеды,Низкие кеды,1139,NaN,//a.lmcdn.ru/product/M/P/MP002XW1FY1D_26634923...,lamoda_photos/Covani_Covani_Кеды__0.jpg,https://a.lmcdn.ru/photoreview/?key=99b534cb-6...,user_photos/Covani_Covani_Кеды__2.jpg
3,Lumiere Magique,Lumiere Magique Кеды Lamoda exclusive,Низкие кеды,76008,NaN,//a.lmcdn.ru/product/M/P/MP002XW1IX7A_27908839...,lamoda_photos/Lumiere_Magique_Lumiere_Magique_...,https://a.lmcdn.ru/photoreview/?key=cd5dc102-a...,user_photos/Lumiere_Magique_Lumiere_Magique_Ке...
4,Tommy Hilfiger,Tommy Hilfiger Кроссовки RUNNER,Низкие кроссовки,86699,NaN,//a.lmcdn.ru/product/R/T/RTLAEV062801_31457487...,lamoda_photos/Tommy_Hilfiger_Tommy_Hilfiger_Кр...,https://a.lmcdn.ru/photoreview/?key=59d21e75-f...,user_photos/Tommy_Hilfiger_Tommy_Hilfiger_Крос...
...,...,...,...,...,...,...,...,...,...
459,Ecco,Ecco Кеды SOFT 7 W,Низкие кеды,547,NaN,//a.lmcdn.ru/product/M/P/MP002XW0S0Z2_11057679...,lamoda_photos/Ecco_Ecco_Кеды_SOFT_7_W_0.jpg,https://a.lmcdn.ru/photoreview/?key=decae862-b...,user_photos/Ecco_Ecco_Кеды_SOFT_7_W_0.jpg
460,Kappa,Kappa Кроссовки SELECTO MD,Кроссовки,93057,"Кроссовки выполнены из синтетической кожи, доп...",//a.lmcdn.ru/product/M/P/MP002XW0OTPY_22083371...,lamoda_photos/Kappa_Kappa_Кроссовки_SELECTO_MD...,https://a.lmcdn.ru/photoreview/?key=70c0ac26-b...,user_photos/Kappa_Kappa_Кроссовки_SELECTO_MD_2...
461,Karl Lagerfeld,Karl Lagerfeld Кеды,Низкие кеды,72503,NaN,//a.lmcdn.ru/product/M/P/MP002XW1DF5T_31481364...,lamoda_photos/Karl_Lagerfeld_Karl_Lagerfeld_Ке...,https://a.lmcdn.ru/photoreview/?key=757e82fc-0...,user_photos/Karl_Lagerfeld_Karl_Lagerfeld_Кеды...
462,Pierre Cardin,Pierre Cardin Кеды,Низкие кеды,53065,NaN,//a.lmcdn.ru/product/M/P/MP002XW1FBNU_26660772...,lamoda_photos/Pierre_Cardin_Pierre_Cardin_Кеды...,https://a.lmcdn.ru/photoreview/?key=a362e9b7-e...,user_photos/Pierre_Cardin_Pierre_Cardin_Кеды__...


In [10]:
test_data = user_data[-100:]
image_reference = dict(zip(test_data["path_to_user_photo"], test_data["model"]))

In [19]:
def count_metrics(
        collection_name: str,
        get_image_embedding: callable):

    recall_5 = 0
    recall_10 = 0
    accuracy = 0
    query_time = []
    N = 0
    unprocessable = []

    for image_path, reference_model in image_reference.items():
        try:
            start = time.perf_counter()
            query = get_image_embedding(prefix_path+image_path)
            candidates = client.query_points(
                                        collection_name=collection_name,
                                        query=query, 
                                        using="photos",
                                        limit=10,           
                                        with_payload=True   # Возвращаем бренд, модель и путь из CSV
                                    ).points
            end = time.perf_counter()
        
            results = [candidate.payload.get("model") for candidate in candidates]
            if reference_model in results:
                recall_10 += 1
            if reference_model in results[:5]:
                recall_5 += 1
            if reference_model == results[0]:
                accuracy += 1
            query_time.append(end-start)
            N +=1

        except:
            unprocessable.append(image_path)
            continue

    print("Unprocessable: ", len(unprocessable))

    return recall_10 *100 / N, recall_5 * 100 / N, accuracy * 100 / N, np.mean(query_time)

In [11]:
lamoda_data_racurses = pd.read_csv(prefix_path+"lamoda_data.csv", sep=";")
lamoda_data_racurses = lamoda_data_racurses.drop_duplicates(subset="title_photo")
print(len(lamoda_data_racurses))

2408


## FT CLIP

In [36]:
from collections import defaultdict

lamoda = defaultdict(list)
for idx, row in lamoda_data_racurses.iterrows():
    lamoda[f'{row["model"]}_{row["color"]}'].append(row["title_photo"])
print("Студийных фото ", len(lamoda))

user = defaultdict(list)
for idx, row in user_data.iterrows():
    user[f'{row["model"]}_{row["color"]}'].append(row["path_to_user_photo"])
print("Пользовательских фото ", len(user))

Студийных фото  407
Пользовательских фото  192


In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments


common_models = set(lamoda.keys()).intersection(set(user.keys()))

pairs = []
for model_id in common_models:
    for user_photo in user[f"{model_id}"]:
        lamoda_photo = random.choice(lamoda[f"{model_id}"])
        pairs.append((user_photo, lamoda_photo))

# 1. Подготовка данных в формате словаря (только пути!)
train_data_dict = {
    "image_1": [prefix_path + p[0] for p in pairs],
    "image_2": [prefix_path + p[1] for p in pairs]
}
hf_dataset = Dataset.from_dict(train_data_dict)

model = SentenceTransformer('clip-ViT-B-32', device=device)
train_loss = losses.MultipleNegativesRankingLoss(model)

# 2. Настройка аргументов обучения
args = SentenceTransformerTrainingArguments(
 output_dir='/home/inna/Рабочий стол/SneakerSearch/models/ft_clip_sneakers',
    num_train_epochs=10,          
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1, 
    warmup_steps=20,
    fp16=True,
    save_steps=100,
    logging_steps=10,
    dataloader_num_workers=2,          
    report_to="none",
)

# 3. Создание тренера
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=hf_dataset,
    loss=train_loss,
)

trainer.train()


Step,Training Loss
10,1.304295
20,1.344899
30,2.642013
40,1.257633
50,1.465570
60,1.387697
70,1.386263
80,1.386249
90,1.386260
100,1.386252


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.26s/it]


TrainOutput(global_step=1150, training_loss=1.3950579676420791, metrics={'train_runtime': 880.0091, 'train_samples_per_second': 5.227, 'train_steps_per_second': 1.307, 'total_flos': 0.0, 'train_loss': 1.3950579676420791, 'epoch': 10.0})

In [ ]:
from sentence_transformers import SentenceTransformer

checkpoint_path = "/home/inna/Рабочий стол/SneakerSearch/models/ft_clip_sneakers/checkpoint-1150"
model = SentenceTransformer(checkpoint_path, device=device)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 19638.29it/s]


In [41]:
def get_image_embedding_clip_ft(img_path):

    image = Image.open(img_path).convert('RGB')

    with torch.no_grad():
        image_emb = model.encode(
            image,
            convert_to_tensor=True,
            normalize_embeddings=True
            )
    
    return image_emb.cpu().numpy().flatten()

In [45]:
collection_name = "Sneakers_CLIP_FT"
emb_dim = 512

create_db(
    client,
    collection_name,
    emb_dim,
    lamoda_data,
    get_image_embedding_clip_ft
)

unprocessable:  0


In [ ]:
recall_10_clip_ft, recall_5_clip_ft, accuracy_clip_ft, avg_query_time_clip_ft = count_metrics(collection_name, get_image_embedding_clip_ft)

pd.DataFrame(
    data={
        "CLIP_FT" : [accuracy_clip_ft, recall_5_clip_ft, recall_10_clip_ft, avg_query_time_clip_ft, 16.4]
    },
    index = ["Accuracy", "Recall@5", "Recall@10", "Avg_query_time, s", "Create_DB_time, s"]
)

Unprocessable:  0


,CLIP_FT
Accuracy,0.000000
Recall@5,1.000000
Recall@10,4.000000
"Avg_query_time, s",0.191015
"Create_DB_time, s",16.400000
